# 🔬 Early Detection of Diabetic Retinopathy using CNN + Explainable AI
**Dataset:** APTOS 2019 Blindness Detection (Kaggle)

## How to use this notebook
1. Add the dataset: sidebar → **Data** → search `aptos2019-blindness-detection`
2. Enable GPU: **Settings → Accelerator → GPU T4**
3. Run all cells top to bottom (**Run All**)

## What this notebook does — cell by cell
| Cell | Purpose |
|------|----------------------------------|
| 1 | Install missing libraries |
| 2 | Imports & global config (edit here) |
| 3 | Load CSVs, verify paths |
| 4 | EDA — class distribution charts |
| 5 | EDA — sample images per grade |
| 6 | Preprocessing functions (Ben Graham, crop) |
| 7 | Augmentation pipelines (train / val) |
| 8 | PyTorch Dataset class |
| 9 | Stratified K-Fold split + weighted sampler |
| 10 | DataLoader factory |
| 11 | EfficientNet-B4 model definition |
| 12 | Focal Loss + class weights |
| 13 | Train / validate step functions |
| 14 | Full training loop (runs fold 0) |
| 15 | Training curves plot |
| 16 | Confusion matrix + classification report |
| 17 | Grad-CAM implementation |
| 18 | Grad-CAM++ implementation |
| 19 | Grad-CAM across all 5 DR grades |
| 20 | Grad-CAM vs Grad-CAM++ comparison |
| 21 | LIME explanation |
| 22 | 4-panel publication figure generator |
| 23 | Summary of all saved outputs |

---
## Cell 1 — Install libraries
Install `lime` and `kagglehub` for dataset download. All other libraries are pre-installed on Kaggle.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install missing libraries                                          ║
# ║  lime  : Local Interpretable Model-agnostic Explanations (XAI method 2)      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip_install('lime')
pip_install('kagglehub')
print('✓ lime installed')
print('✓ kagglehub installed')

# Verify key libraries are available
import torch, timm, albumentations
print(f'✓ PyTorch  {torch.__version__}')
print(f'✓ timm     {timm.__version__}')
print(f'✓ albumentations {albumentations.__version__}')
print(f'✓ CUDA available: {torch.cuda.is_available()}')

---
## Cell 2 — Imports & global config
**All tunable hyperparameters live here.** Change `CFG` to experiment:
- Increase `img_size` to `380` for better accuracy (slower)
- Change `backbone` to `'resnet50'` or `'densenet121'` for baseline comparison
- Set `use_focal_loss: False` to compare against plain weighted cross-entropy
- Remove the `break` in Cell 14 to train all 5 folds

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports & global config                                          ║
# ║  Edit CFG to change hyperparameters. Everything else reads from CFG.       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, cv2, time, copy, random, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    cohen_kappa_score, confusion_matrix, classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from lime import lime_image
from skimage.segmentation import mark_boundaries

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Global config — edit these to experiment ──────────────────────────────────
CFG = {
    # Data
    'img_size'       : 224,           # 224 = fast; 380 = best accuracy
    'batch_size'     : 32,
    'num_workers'    : 4,
    'n_folds'        : 5,
    'seed'           : SEED,

    # Model
    'backbone'       : 'efficientnet_b4',  # try 'resnet50', 'densenet121'
    'num_classes'    : 5,
    'dropout'        : 0.3,

    # Training
    'epochs'         : 20,
    'lr'             : 1e-4,
    'weight_decay'   : 1e-5,
    'patience'       : 5,            # early stopping — stop if no QWK gain

    # Loss
    'use_focal_loss' : True,         # False = plain weighted cross-entropy
    'focal_gamma'    : 2.0,          # higher = more focus on hard/rare samples

    # Paths
    'output_dir' : Path('/kaggle/working'),
}

# ── Download dataset via kagglehub ────────────────────────────────────────────
import kagglehub
_dl_path = kagglehub.competition_download('aptos2019-blindness-detection')
CFG['data_dir'] = Path(_dl_path)
print(f'Dataset path: {_dl_path}')

# ── Derived paths ─────────────────────────────────────────────────────────────
TRAIN_IMG = CFG['data_dir'] / 'train_images'
TEST_IMG  = CFG['data_dir'] / 'test_images'
TRAIN_CSV = CFG['data_dir'] / 'train.csv'
TEST_CSV  = CFG['data_dir'] / 'test.csv'
CKPT_DIR  = CFG['output_dir'] / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Labels ────────────────────────────────────────────────────────────────────
LABEL_MAP = {
    0: 'No DR',
    1: 'Mild',
    2: 'Moderate',
    3: 'Severe',
    4: 'Proliferative'
}

# ImageNet normalisation stats (used because we load pretrained weights)
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

print(f'Device  : {DEVICE}')
print(f'Backbone: {CFG["backbone"]}')
print(f'Img size: {CFG["img_size"]}×{CFG["img_size"]}')
print(f'Loss    : {"Focal" if CFG["use_focal_loss"] else "Weighted CrossEntropy"}')
print(f'Data dir     : {CFG["data_dir"]}')
print(f'Data dir exists: {CFG["data_dir"].exists()}')
print(f'train_images exists: {TRAIN_IMG.exists()}')
print(f'train.csv    exists: {TRAIN_CSV.exists()}')

---
## Cell 3 — Load CSVs and verify paths
Reads `train.csv` (id_code + diagnosis columns) and builds full image paths.
Also verifies a sample of images can actually be opened — catches common Kaggle
dataset attachment issues early.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load CSVs & verify paths                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Attach full paths as a column
train_df['path'] = train_df['id_code'].apply(
    lambda x: str(TRAIN_IMG / f'{x}.png')
)
test_df['path'] = test_df['id_code'].apply(
    lambda x: str(TEST_IMG / f'{x}.png')
)
train_df['label_name'] = train_df['diagnosis'].map(LABEL_MAP)

print(f'Train samples : {len(train_df)}')
print(f'Test  samples : {len(test_df)}')

# Sanity-check: confirm 10 images are readable
bad_paths = []
for path in train_df['path'].sample(10, random_state=42):
    if cv2.imread(path) is None:
        bad_paths.append(path)

if bad_paths:
    print(f'⚠ WARNING: {len(bad_paths)} unreadable images — check dataset attachment')
    for p in bad_paths: print(' ', p)
else:
    print('✓ All sampled images readable')

display(train_df.head())

---
## Cell 4 — EDA: class distribution
APTOS 2019 has **severe class imbalance** — Grade 0 (No DR) accounts for ~49%
of all samples while Grade 3 (Severe) is only ~6%. This is the core problem
this project addresses through Focal Loss and weighted sampling.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — EDA: class distribution                                          ║
# ║  Output: class_distribution.png                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

counts = train_df['diagnosis'].value_counts().sort_index()
pcts   = counts / counts.sum() * 100

print('Class distribution:')
for k, v in counts.items():
    bar = '█' * int(pcts[k])
    print(f'  Grade {k} ({LABEL_MAP[k]:>13s}): {v:>4d}  ({pcts[k]:.1f}%)  {bar}')
print(f'\nImbalance ratio (No DR ÷ Proliferative): {counts[0]/counts[4]:.1f}×')

COLORS = ['#4CAF50','#FFC107','#FF9800','#F44336','#9C27B0']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar([LABEL_MAP[i] for i in range(5)], counts.values,
            color=COLORS, edgecolor='white', linewidth=1.2)
axes[0].set_title('Class distribution — count', fontsize=13)
axes[0].set_ylabel('Number of images')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values,
            labels=[LABEL_MAP[i] for i in range(5)],
            colors=COLORS, autopct='%1.1f%%', startangle=140,
            textprops={'fontsize': 11})
axes[1].set_title('Class distribution — percentage', fontsize=13)

plt.suptitle('APTOS 2019 — severe class imbalance\n'
             'Addressed via Focal Loss + Weighted Random Sampler',
             fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'class_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 5 — EDA: sample images per grade
Shows 5 random samples from each of the 5 DR severity grades so you can
visually understand what the model needs to distinguish. Grade 0 shows
a clean retina; Grade 4 shows neovascularisation and large haemorrhages.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — EDA: sample retinal images per DR grade                         ║
# ║  Output: sample_images.png                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def quick_load(path, size=224):
    """Fast loader just for EDA display — no preprocessing applied."""
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (size, size))

fig, axes = plt.subplots(5, 5, figsize=(16, 17))

for grade in range(5):
    subset  = train_df[train_df['diagnosis'] == grade]
    samples = subset.sample(min(5, len(subset)), random_state=SEED)

    for col, (_, row) in enumerate(samples.iterrows()):
        ax = axes[grade][col]
        ax.imshow(quick_load(row['path']))
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(
                f"Grade {grade}\n{LABEL_MAP[grade]}",
                fontsize=12, rotation=0,
                labelpad=85, va='center', color=COLORS[grade]
            )
        ax.set_title(row['id_code'], fontsize=7, pad=2)

plt.suptitle('Sample fundus images — 5 severity grades × 5 images',
             fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'sample_images.png',
            dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 6 — Preprocessing functions
Two preprocessing steps applied to every image before training and inference:
1. **`crop_black_border`** — removes the dark vignette borders that fundus
   cameras introduce. Without this the model wastes capacity learning that
   black corners mean nothing.
2. **`ben_graham`** — subtracts the local Gaussian blur from the image
   (`result = 4×img − 4×blur + 128`). This equalises the brightness across
   the retina and sharpens microaneurysms and haemorrhages. It was the
   single most impactful preprocessing trick in the original 2015 Kaggle DR
   competition (used by top solutions).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Preprocessing functions                                          ║
# ║  crop_black_border : remove dark vignette borders                          ║
# ║  ben_graham        : local contrast normalisation                          ║
# ║  load_and_preprocess: full pipeline used by the Dataset class              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def crop_black_border(img, tol=7):
    """
    Remove rows/columns where the mean pixel value is <= tol.
    Fundus cameras produce circular images on a black background;
    cropping removes the useless border and gives the network more
    resolution to focus on the retina.
    """
    mask = img.mean(axis=2) > tol
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    if rows.size == 0 or cols.size == 0:
        return img   # image was all black — return unchanged
    return img[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]


def ben_graham(img, sigmaX=10):
    """
    Ben Graham's local-mean subtraction.
    Formula:  out = 4*img - 4*GaussianBlur(img) + 128
    This removes low-frequency illumination variation and boosts
    the contrast of fine structures (microaneurysms, exudates).
    The circular mask zeros out the background to prevent
    edge artefacts from the Gaussian filter.
    """
    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX)
    result  = cv2.addWeighted(img, 4, blurred, -4, 128)

    # Circular mask: keep only the retinal disc area
    mask = np.zeros(img.shape, dtype=np.float32)
    cv2.circle(
        mask,
        (img.shape[1] // 2, img.shape[0] // 2),
        int(img.shape[0] * 0.9 // 2),
        (1, 1, 1), -1, 8, 0
    )
    result = result * mask + 128 * (1 - mask)
    return result.astype(np.uint8)


def load_and_preprocess(path, img_size=224, use_ben_graham=True):
    """
    Full preprocessing pipeline:
      1. Read as RGB
      2. Crop black border
      3. Resize to img_size × img_size
      4. Apply Ben Graham (optional — set False to compare)
    Returns uint8 numpy array (H, W, 3).
    """
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f'Cannot read image: {path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = crop_black_border(img)
    img = cv2.resize(img, (img_size, img_size))
    if use_ben_graham:
        img = ben_graham(img)
    return img


# ── Visual comparison: raw vs preprocessed ────────────────────────────────────
sample_path = train_df[train_df['diagnosis'] == 2].iloc[0]['path']
raw  = load_and_preprocess(sample_path, CFG['img_size'], use_ben_graham=False)
proc = load_and_preprocess(sample_path, CFG['img_size'], use_ben_graham=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(raw);  axes[0].set_title('Raw (after crop + resize)');          axes[0].axis('off')
axes[1].imshow(proc); axes[1].set_title('After Ben Graham preprocessing');     axes[1].axis('off')
plt.suptitle('Ben Graham contrast enhancement — microaneurysms become clearer',
             fontsize=12)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'preprocessing_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Preprocessing functions ready')

---
## Cell 7 — Augmentation pipelines
**Training augmentations** apply random geometric and photometric transforms
to each image before it enters the network. This artificially expands the
effective dataset size and reduces overfitting.

**Validation augmentations** only normalise — no random transforms,
so validation metrics are reproducible.

We use **albumentations** (faster than torchvision transforms for image data).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Augmentation pipelines (train / val)                             ║
# ║  Uses albumentations for fast CPU-side augmentation.                       ║
# ║  All transforms operate on uint8 (H,W,3) numpy arrays.                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

train_transforms = A.Compose([
    # ── Geometric transforms ───────────────────────────────────────────────────
    A.HorizontalFlip(p=0.5),             # retina has no preferred orientation
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=30, p=0.5),           # slight tilt variation
    A.ShiftScaleRotate(
        shift_limit=0.05, scale_limit=0.1,
        rotate_limit=0, p=0.3
    ),

    # ── Photometric transforms ─────────────────────────────────────────────────
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2, p=0.5
    ),
    A.HueSaturationValue(
        hue_shift_limit=10, sat_shift_limit=20,
        val_shift_limit=10, p=0.3
    ),

    # ── Noise / occlusion ─────────────────────────────────────────────────────
    A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
    A.CoarseDropout(
        num_holes_range=(1, 8), hole_height_range=(8, 16),
        hole_width_range=(8, 16), fill=0, p=0.3
    ),  # simulates small lesion occlusions

    # ── Normalise to ImageNet stats + convert to float tensor ─────────────────
    A.Normalize(mean=MEAN.tolist(), std=STD.tolist()),
    ToTensorV2(),    # (H,W,C) uint8  →  (C,H,W) float32
])

val_transforms = A.Compose([
    # No random transforms — only normalise for reproducible evaluation
    A.Normalize(mean=MEAN.tolist(), std=STD.tolist()),
    ToTensorV2(),
])

# ── Quick shape check ─────────────────────────────────────────────────────────
test_img_np = load_and_preprocess(train_df.iloc[0]['path'], CFG['img_size'])
test_tensor = train_transforms(image=test_img_np)['image']
print(f'Input  shape: {test_img_np.shape}  dtype: {test_img_np.dtype}')
print(f'Output shape: {test_tensor.shape}  dtype: {test_tensor.dtype}')
print(f'Value range : {test_tensor.min():.2f} → {test_tensor.max():.2f}')
print(f'✓ {len(train_transforms.transforms)} train transforms, '
      f'{len(val_transforms.transforms)} val transforms')

---
## Cell 8 — PyTorch Dataset class
`RetinalDataset` wraps the DataFrame into a standard `torch.utils.data.Dataset`.
For each index it:
1. Reads the image path from the dataframe row
2. Calls `load_and_preprocess` (Cell 6)
3. Applies the albumentations transform (Cell 7)
4. Returns `(tensor, label)` for training or just `tensor` for test

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — PyTorch Dataset class                                            ║
# ║  RetinalDataset: wraps a DataFrame into a torch-compatible Dataset.        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class RetinalDataset(Dataset):
    """
    Parameters
    ----------
    df            : DataFrame with columns 'path' and (optionally) 'diagnosis'
    img_size      : resize target (should match CFG['img_size'])
    transform     : albumentations Compose pipeline
    use_ben_graham: whether to apply Ben Graham preprocessing
    is_test       : if True, __getitem__ returns only the image (no label)
    """
    def __init__(self, df, img_size=224, transform=None,
                 use_ben_graham=True, is_test=False):
        self.df             = df.reset_index(drop=True)
        self.img_size       = img_size
        self.transform      = transform
        self.use_ben_graham = use_ben_graham
        self.is_test        = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = load_and_preprocess(
            row['path'], self.img_size, self.use_ben_graham
        )
        if self.transform:
            img = self.transform(image=img)['image']

        if self.is_test:
            return img
        return img, torch.tensor(row['diagnosis'], dtype=torch.long)


# ── Verify dataset output ─────────────────────────────────────────────────────
dummy_ds = RetinalDataset(train_df.head(4), CFG['img_size'], val_transforms)
img_t, lbl_t = dummy_ds[0]
print(f'Image tensor shape : {img_t.shape}')   # expected: (3, 224, 224)
print(f'Label tensor       : {lbl_t.item()} = {LABEL_MAP[lbl_t.item()]}')
print('✓ RetinalDataset ready')

---
## Cell 9 — Stratified K-Fold split + weighted sampler
**StratifiedKFold** ensures each fold has the same class ratio as the full
dataset — critical when some classes have very few samples.

**WeightedRandomSampler** over-samples minority classes at the batch level
so each mini-batch the model sees during training contains a roughly balanced
number of images from each severity grade. This works alongside Focal Loss.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Stratified K-Fold + WeightedRandomSampler                       ║
# ║  SKF: each fold has the same class proportions as the full dataset.        ║
# ║  WRS: minority classes are over-sampled at the batch level.                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Assign fold IDs ───────────────────────────────────────────────────────────
skf = StratifiedKFold(
    n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed']
)
train_df['fold'] = -1
for fold_id, (_, val_idx) in enumerate(
    skf.split(train_df, train_df['diagnosis'])
):
    train_df.loc[val_idx, 'fold'] = fold_id

print('Fold distribution (rows=fold, cols=grade):')
print(train_df.groupby(['fold','diagnosis']).size().unstack().to_string())


# ── Weighted random sampler factory ───────────────────────────────────────────
def make_weighted_sampler(labels):
    """
    Compute per-sample weights inversely proportional to class frequency.
    A class with 100 samples gets weight 1/100; one with 10 gets weight 1/10.
    Result: each class is equally likely to appear in a random batch.
    """
    class_counts  = np.bincount(labels)
    class_weights = 1.0 / class_counts               # inverse frequency
    sample_weights = class_weights[labels]            # per-sample weight
    return WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True    # allows rare classes to repeat within an epoch
    )

print('\n✓ Fold assignment + sampler factory ready')

---
## Cell 10 — DataLoader factory
`get_dataloaders(fold)` returns two DataLoaders for a given fold:
- **train_loader**: uses weighted sampler + `train_transforms`
- **val_loader**: plain sequential sampler + `val_transforms`

`pin_memory=True` speeds up GPU data transfer on Kaggle's T4 instances.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — DataLoader factory                                              ║
# ║  get_dataloaders(fold) builds train + val loaders for one CV fold.         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def get_dataloaders(fold, cfg=CFG):
    """
    Returns (train_loader, val_loader) for the given fold index.
    Training uses WeightedRandomSampler to balance class frequency.
    Validation uses sequential order for reproducibility.
    """
    tr_df = train_df[train_df['fold'] != fold].copy()
    vl_df = train_df[train_df['fold'] == fold].copy()

    tr_ds = RetinalDataset(tr_df, cfg['img_size'], train_transforms)
    vl_ds = RetinalDataset(vl_df, cfg['img_size'], val_transforms)

    sampler = make_weighted_sampler(tr_df['diagnosis'].values)

    tr_loader = DataLoader(
        tr_ds,
        batch_size  = cfg['batch_size'],
        sampler     = sampler,           # weighted — no shuffle kwarg
        num_workers = cfg['num_workers'],
        pin_memory  = True
    )
    vl_loader = DataLoader(
        vl_ds,
        batch_size  = cfg['batch_size'],
        shuffle     = False,
        num_workers = cfg['num_workers'],
        pin_memory  = True
    )
    print(f'Fold {fold}  |  train: {len(tr_ds)} samples  |  val: {len(vl_ds)} samples')
    return tr_loader, vl_loader

# ── Test fold 0 ───────────────────────────────────────────────────────────────
tr_loader, vl_loader = get_dataloaders(fold=0)
imgs, labels = next(iter(tr_loader))
print(f'Batch shape : {imgs.shape}')       # (32, 3, 224, 224)
print(f'Label sample: {labels[:8].tolist()}')
print('✓ DataLoaders ready')

---
## Cell 11 — Model: EfficientNet-B4 + custom head
We use **EfficientNet-B4** pretrained on ImageNet via the `timm` library.
The original classification head is removed (`num_classes=0`) and replaced
with our own `Dropout → Linear(5)` head.

During training we use **two learning rates** (discriminative fine-tuning):
- Backbone layers: `lr × 0.1` — pretrained weights change slowly
- Head layers: `lr` — new weights learn faster

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Model definition                                                ║
# ║  DRModel: EfficientNet-B4 backbone + dropout + linear classification head  ║
# ║  timm.create_model downloads pretrained ImageNet weights automatically.    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class DRModel(nn.Module):
    """
    Diabetic Retinopathy classifier.

    Architecture:
        EfficientNet-B4 (pretrained, head removed)
            ↓  global average pool  →  (B, 1792)
        Dropout(p=dropout)
        Linear(1792 → num_classes)

    Dropout before the final linear layer acts as regularisation
    against overfitting on the small APTOS dataset (~3 k images).
    """
    def __init__(self, backbone='efficientnet_b4',
                 num_classes=5, pretrained=True, dropout=0.3):
        super().__init__()
        # Load backbone without the original ImageNet head
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained, num_classes=0
        )
        in_features = self.backbone.num_features  # 1792 for EfficientNet-B4

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)   # (B, in_features)
        return self.head(features)    # (B, num_classes)


# ── Architecture sanity check ─────────────────────────────────────────────────
model = DRModel(
    CFG['backbone'], CFG['num_classes'],
    pretrained=True, dropout=CFG['dropout']
).to(DEVICE)

with torch.no_grad():
    dummy  = torch.randn(2, 3, CFG['img_size'], CFG['img_size']).to(DEVICE)
    output = model(dummy)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Output shape     : {output.shape}')        # (2, 5)
print(f'Total params     : {total_params/1e6:.1f}M')
print(f'Trainable params : {trainable_params/1e6:.1f}M')
print('✓ Model ready')

---
## Cell 12 — Focal Loss + class weights
**Focal Loss** (Lin et al., 2017) modifies standard cross-entropy by
adding a `(1 − p_t)^γ` modulating factor. When the model is confident
(`p_t` is high), the factor is near 0 and the loss contribution is suppressed.
When the model struggles (`p_t` is low), the factor is near 1 and the
loss is unchanged. This focuses training on the rare, hard-to-classify
cases (Severe, Proliferative DR).

We also pass `weight=class_weights` so the loss is additionally scaled
by inverse class frequency — double protection against imbalance.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Focal Loss + class weights                                      ║
# ║  FocalLoss: Lin et al. 2017 (https://arxiv.org/abs/1708.02002)             ║
# ║  class_weights: inverse frequency — minority classes penalised more.       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class FocalLoss(nn.Module):
    """
    Focal loss for multi-class classification.

    Loss = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Parameters
    ----------
    gamma   : focusing parameter. 0 = standard CE; 2 is the paper default.
    weight  : class weights tensor (inverse frequency). Same as CE's weight arg.
    reduction: 'mean' or 'sum'
    """
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.weight    = weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Standard per-sample cross-entropy (unreduced)
        ce_loss = F.cross_entropy(
            inputs, targets,
            weight    = self.weight,
            reduction = 'none'
        )
        # p_t: probability assigned to the correct class
        pt = torch.exp(-ce_loss)

        # Focal modulating factor
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()


# ── Compute class weights from full training set ───────────────────────────────
class_counts  = np.bincount(train_df['diagnosis'].values)
class_weights = 1.0 / class_counts                          # inverse frequency
class_weights = class_weights / class_weights.sum() * CFG['num_classes']  # normalise
class_weights_t = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

# ── Instantiate loss function ─────────────────────────────────────────────────
if CFG['use_focal_loss']:
    criterion = FocalLoss(gamma=CFG['focal_gamma'], weight=class_weights_t)
    print(f'Loss: Focal Loss  (γ={CFG["focal_gamma"]})')
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)
    print('Loss: Weighted CrossEntropy')

print('Class weights:', dict(zip(LABEL_MAP.values(),
                                  class_weights_t.cpu().numpy().round(3))))

---
## Cell 13 — Train and validate step functions
`train_one_epoch` runs one full pass through the training DataLoader with
**mixed-precision training** (`autocast + GradScaler`) — this halves GPU
memory usage and speeds up training by ~30% on T4 GPUs.

`validate` runs inference with `torch.no_grad()` and computes
**Quadratic Weighted Kappa (QWK)** — the official APTOS competition metric
and the one clinicians care about (penalises large misgrading more than
adjacent-grade errors).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — train_one_epoch & validate                                      ║
# ║  Mixed precision (autocast) for memory efficiency and speed.               ║
# ║  validate() returns QWK (quadratic weighted kappa) — key metric.           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    """
    One training epoch. Returns (avg_loss, accuracy).
    Uses mixed-precision via GradScaler to reduce GPU memory usage.
    Gradient clipping (max_norm=1.0) prevents exploding gradients.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast(device_type='cuda'):      # FP16 forward pass (PyTorch 2.x)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()           # scaled backward
        scaler.unscale_(optimizer)              # unscale before clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    """
    Validation pass. Returns (avg_loss, QWK, accuracy, preds_list, labels_list).
    QWK = Quadratic Weighted Kappa — punishes grade 0→4 errors more than 0→1.
    Range: −1 (worse than chance) to 1 (perfect). Target: > 0.90.
    """
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast(device_type='cuda'):      # FP16 (PyTorch 2.x)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    qwk      = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))

    return avg_loss, qwk, accuracy, all_preds, all_labels


print('✓ train_one_epoch and validate functions ready')

---
## Cell 14 — Full training loop
Trains fold 0 by default. The `break` at the bottom prevents running
all 5 folds — **remove it** once you've confirmed the model converges.

Key features:
- **Discriminative learning rates** — backbone at `lr/10`, head at `lr`
- **Cosine annealing with warm restarts** — decays LR from `lr` to `1e-6`,
  then resets every 10 epochs
- **Early stopping** — halts training if QWK doesn't improve for `patience` epochs
- **Checkpoint saving** — saves the best QWK model weights per fold

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Training loop (5-fold cross-validation)                        ║
# ║  Currently runs fold 0 only (has a break at the end).                     ║
# ║  Remove the break to train all folds (~30–60 min/fold on Kaggle T4 GPU).  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def train_fold(fold, criterion, cfg=CFG):
    print(f'\n{"═"*62}')
    print(f'  FOLD {fold+1} / {cfg["n_folds"]}   backbone={cfg["backbone"]}')
    print(f'{"═"*62}')

    tr_loader, vl_loader = get_dataloaders(fold, cfg)

    # Fresh model for each fold — prevents fold leakage
    mdl = DRModel(
        cfg['backbone'], cfg['num_classes'],
        pretrained=True, dropout=cfg['dropout']
    ).to(DEVICE)

    scaler = GradScaler()

    # Discriminative learning rates:
    # backbone gets 10× lower LR to preserve pretrained features
    optimizer = optim.AdamW([
        {'params': mdl.backbone.parameters(), 'lr': cfg['lr'] * 0.1},
        {'params': mdl.head.parameters(),     'lr': cfg['lr']},
    ], weight_decay=cfg['weight_decay'])

    # Cosine annealing: LR decays from lr → eta_min then resets every T_0 epochs
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=1, eta_min=1e-6
    )

    best_qwk       = -np.inf
    patience_count = 0
    history = {'tr_loss':[], 'vl_loss':[], 'vl_qwk':[], 'vl_acc':[]}

    for epoch in range(1, cfg['epochs'] + 1):
        t0 = time.time()

        tr_loss, tr_acc = train_one_epoch(
            mdl, tr_loader, optimizer, criterion, scaler, DEVICE
        )
        vl_loss, vl_qwk, vl_acc, preds, labels = validate(
            mdl, vl_loader, criterion, DEVICE
        )
        scheduler.step()

        history['tr_loss'].append(tr_loss)
        history['vl_loss'].append(vl_loss)
        history['vl_qwk'].append(vl_qwk)
        history['vl_acc'].append(vl_acc)

        elapsed  = time.time() - t0
        lr_now   = optimizer.param_groups[1]['lr']   # head LR
        improved = '✓' if vl_qwk > best_qwk else ' '

        print(f'[{improved}] E{epoch:02d}/{cfg["epochs"]}  '
              f'tr={tr_loss:.4f}/{tr_acc:.3f}  '
              f'vl={vl_loss:.4f}  '
              f'QWK={vl_qwk:.4f}  '
              f'acc={vl_acc:.3f}  '
              f'lr={lr_now:.2e}  [{elapsed:.0f}s]')

        if vl_qwk > best_qwk:
            best_qwk       = vl_qwk
            patience_count = 0
            ckpt_path      = CKPT_DIR / f'best_fold{fold}.pth'
            torch.save({
                'epoch'      : epoch,
                'state_dict' : mdl.state_dict(),
                'best_qwk'   : best_qwk,
                'preds'      : preds,
                'labels'     : labels,
                'cfg'        : cfg,
            }, ckpt_path)
        else:
            patience_count += 1
            if patience_count >= cfg['patience']:
                print(f'  Early stopping — no improvement for {cfg["patience"]} epochs.')
                break

    print(f'\n  Fold {fold} best QWK: {best_qwk:.4f}  saved → {ckpt_path}')
    return history, best_qwk


# ── Run cross-validation ──────────────────────────────────────────────────────
all_histories = {}
all_qwks      = []

for fold in range(CFG['n_folds']):
    history, best_qwk = train_fold(fold, criterion)
    all_histories[fold] = history
    all_qwks.append(best_qwk)
    break  # ← REMOVE THIS LINE to train all 5 folds

print(f'\nQWK per fold : {[round(q,4) for q in all_qwks]}')
print(f'Mean ± std   : {np.mean(all_qwks):.4f} ± {np.std(all_qwks):.4f}')

---
## Cell 15 — Training curves
Plots loss, QWK, and accuracy over epochs for each trained fold.
Useful for spotting overfitting (val loss rising while train loss falls)
and confirming early stopping triggered at the right point.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Training curves                                                 ║
# ║  Output: training_curves_fold{N}.png for each trained fold.                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def plot_training_curves(history, fold):
    """
    3-panel plot:
    Left  : train vs val loss — checks for overfitting
    Centre: val QWK over epochs — the metric we optimise
    Right : val accuracy — easier to interpret at a glance
    """
    epochs = range(1, len(history['tr_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # Loss
    axes[0].plot(epochs, history['tr_loss'], label='Train', linewidth=2)
    axes[0].plot(epochs, history['vl_loss'], label='Val',   linewidth=2)
    axes[0].set_title(f'Fold {fold} — Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend()

    # QWK
    best_epoch = np.argmax(history['vl_qwk']) + 1
    best_val   = max(history['vl_qwk'])
    axes[1].plot(epochs, history['vl_qwk'], color='#E65100',
                 marker='o', ms=4, linewidth=2)
    axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.7,
                    label=f'Best E{best_epoch}: QWK={best_val:.4f}')
    axes[1].set_title(f'Fold {fold} — Quadratic Weighted Kappa')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('QWK')
    axes[1].legend()

    # Accuracy
    axes[2].plot(epochs, history['vl_acc'], color='#2E7D32',
                 marker='s', ms=4, linewidth=2)
    axes[2].set_title(f'Fold {fold} — Validation Accuracy')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')

    plt.tight_layout()
    save_path = CFG['output_dir'] / f'training_curves_fold{fold}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

for fold, history in all_histories.items():
    plot_training_curves(history, fold)

---
## Cell 16 — Confusion matrix + classification report
Loads the best checkpoint from fold 0 (already contains saved predictions)
and produces:
1. **Count confusion matrix** — raw misclassification counts
2. **Normalised confusion matrix** — shows recall per class
3. **Classification report** — precision, recall, F1 per grade

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 16 — Confusion matrix + per-class metrics                            ║
# ║  Output: confusion_matrix.png                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

ckpt = torch.load(CKPT_DIR / 'best_fold0.pth', map_location='cpu', weights_only=False)
preds_best  = ckpt['preds']
labels_best = ckpt['labels']
class_names = [LABEL_MAP[i] for i in range(5)]

cm      = confusion_matrix(labels_best, preds_best)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion matrix (counts)', fontsize=13)
axes[0].set_ylabel('True label', fontsize=11)
axes[0].set_xlabel('Predicted label', fontsize=11)

# Normalised (recall per class = diagonal of cm_norm)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, vmin=0, vmax=1,
            cbar_kws={'label': 'Recall'})
axes[1].set_title('Confusion matrix (normalised — diagonal = recall)',
                  fontsize=13)
axes[1].set_ylabel('True label', fontsize=11)
axes[1].set_xlabel('Predicted label', fontsize=11)

plt.suptitle(f'Fold 0  |  QWK = {ckpt["best_qwk"]:.4f}', fontsize=14)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'confusion_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('\n── Classification Report ─────────────────────────────────────────────')
print(classification_report(labels_best, preds_best,
                             target_names=class_names, digits=4))
print(f'Quadratic Weighted Kappa (fold 0): {ckpt["best_qwk"]:.4f}')

---
## Cell 17 — Grad-CAM implementation
**Gradient-weighted Class Activation Mapping** (Selvaraju et al., 2017).

How it works:
1. Register a hook on the last convolutional layer to capture activations
2. Forward pass → compute class score for target class
3. Backward pass → compute gradients of that score w.r.t. the activations
4. Weight each feature map by its gradient (global average pooled)
5. Sum and ReLU → a coarse heatmap showing which pixels influenced the prediction

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 17 — Grad-CAM                                                        ║
# ║  Selvaraju et al. 2017 https://arxiv.org/abs/1610.02391                    ║
# ║  Produces a (H, W) heatmap showing which retinal region drove prediction.  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Reload model from best checkpoint ────────────────────────────────────────
xai_model = DRModel(
    CFG['backbone'], CFG['num_classes'],
    pretrained=False, dropout=CFG['dropout']
).to(DEVICE)
xai_model.load_state_dict(torch.load(
    CKPT_DIR / 'best_fold0.pth', map_location=DEVICE, weights_only=False
)['state_dict'])
xai_model.eval()


# ── Helper: numpy image → normalised tensor ───────────────────────────────────
def img_to_tensor(img_np):
    """uint8 (H,W,3) RGB array → (1,3,H,W) float32 tensor on DEVICE."""
    x = img_np.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return torch.from_numpy(x).permute(2,0,1).unsqueeze(0).float().to(DEVICE)


class GradCAM:
    """
    Usage:
        cam = GradCAM(model, target_layer)
        heatmap, pred_class, probs = cam.generate(tensor)
    """
    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None

        # Hooks capture the layer's output (forward) and its gradients (backward)
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        """
        Parameters
        ----------
        input_tensor : (1, 3, H, W) float tensor
        class_idx    : which class to explain. None = predicted class.

        Returns
        -------
        cam        : (H, W) numpy array in [0, 1]
        class_idx  : int — which class was explained
        probs      : (num_classes,) numpy array — predicted probabilities
        """
        self.model.zero_grad()
        output = self.model(input_tensor)              # (1, 5)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        # Scalar backward on the target class logit
        output[0, class_idx].backward()

        # Global-average-pool the gradients to get per-channel importance
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        cam     = (weights * self.activations).sum(dim=1).squeeze()  # (H, W)
        cam     = F.relu(cam)                 # negative activations discarded

        if cam.max() > 0:
            cam = cam / cam.max()             # normalise to [0, 1]

        probs = output.softmax(dim=1).cpu().detach().numpy()[0]
        return cam.cpu().numpy(), class_idx, probs


def overlay_heatmap(img_np, cam, alpha=0.45):
    """
    Blend a jet-coloured Grad-CAM heatmap on top of the original image.
    alpha controls heatmap opacity (0=invisible, 1=opaque).
    """
    cam_resized = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
    heatmap     = cv2.applyColorMap(
        (cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET
    )
    heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay     = (alpha * heatmap_rgb + (1 - alpha) * img_np).astype(np.uint8)
    return overlay, cam_resized


# ── Attach Grad-CAM to the last convolutional block of EfficientNet-B4 ────────
# timm EfficientNet-B4: conv_pwl is the last pointwise Conv2d.
# Must hook a Conv2d with spatial H×W output — NOT bn2 (BatchNorm has no
# spatial gradients useful for Grad-CAM).
target_layer = xai_model.backbone.blocks[-1][-1].conv_pwl
gradcam      = GradCAM(xai_model, target_layer)

print(f'Grad-CAM attached to: {type(target_layer).__name__}')
print('✓ Grad-CAM ready')

---
## Cell 18 — Grad-CAM++ implementation
**Grad-CAM++** (Chattopadhyay et al., 2018) improves on Grad-CAM by using
second-order gradients to weight each spatial location differently.
It gives better localisation of **small, localised lesions** like
microaneurysms — which is clinically important for early-stage DR.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 18 — Grad-CAM++                                                      ║
# ║  Chattopadhyay et al. 2018 https://arxiv.org/abs/1710.11063               ║
# ║  Better at localising small lesions than standard Grad-CAM.               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class GradCAMPP:
    """
    Grad-CAM++ uses second-order gradient information to assign
    pixel-wise importance weights, giving crisper localisation
    of small retinal lesions compared to standard Grad-CAM.
    """
    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        output[0, class_idx].backward()

        grads = self.gradients               # (1, C, H, W)
        acts  = self.activations             # (1, C, H, W)

        # Second-order gradient weighting
        grads_sq  = grads ** 2
        grads_cub = grads ** 3
        denom     = 2 * grads_sq + acts * grads_cub
        denom     = torch.where(denom != 0, denom, torch.ones_like(denom))
        alpha     = grads_sq / denom

        # Only use ReLU-filtered gradients
        relu_grads = F.relu(output[0, class_idx].exp() * grads)
        weights    = (alpha * relu_grads).sum(dim=[2, 3], keepdim=True)

        cam = (weights * acts).sum(dim=1).squeeze()
        cam = F.relu(cam)
        if cam.max() > 0:
            cam = cam / cam.max()

        probs = output.softmax(dim=1).cpu().detach().numpy()[0]
        return cam.cpu().numpy(), class_idx, probs


# Fresh hook on the same conv_pwl layer (correct Conv2d for Grad-CAM++)
target_layer_pp = xai_model.backbone.blocks[-1][-1].conv_pwl
gradcampp = GradCAMPP(xai_model, target_layer_pp)
print('✓ Grad-CAM++ ready')

---
## Cell 19 — Grad-CAM across all 5 DR grades
The main explainability figure for the paper. Shows one representative retinal
image from each grade with its Grad-CAM overlay and the model's confidence
scores across all 5 classes.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 19 — Grad-CAM visualisation across all 5 DR grades                  ║
# ║  3 columns: original | heatmap | overlay + confidence scores              ║
# ║  Output: gradcam_all_grades.png                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(5, 3, figsize=(14, 23))

# Column headers
for col, title in enumerate(['Original fundus image',
                              'Grad-CAM heatmap',
                              'Overlay + confidence']):
    axes[0][col].set_title(title, fontsize=13, fontweight='bold', pad=10)

for grade in range(5):
    # Pick first image of each grade from the validation set
    row_sample = train_df[
        (train_df['diagnosis'] == grade) &
        (train_df['fold'] == 0)
    ].iloc[0]

    img_np = load_and_preprocess(row_sample['path'], CFG['img_size'])
    tensor = img_to_tensor(img_np)

    cam, pred_class, probs = gradcam.generate(tensor)
    overlay, heatmap_2d    = overlay_heatmap(img_np, cam)

    # Column 0: original
    axes[grade][0].imshow(img_np)
    axes[grade][0].set_ylabel(
        f'True: Grade {grade}\n{LABEL_MAP[grade]}\n\nPred: Grade {pred_class}\n{LABEL_MAP[pred_class]}',
        fontsize=9, rotation=0, labelpad=100, va='center',
        color=COLORS[pred_class]
    )

    # Column 1: raw heatmap
    im = axes[grade][1].imshow(heatmap_2d, cmap='jet', vmin=0, vmax=1)

    # Column 2: overlay + confidence bar as xlabel
    axes[grade][2].imshow(overlay)
    conf_str = '  '.join([f'G{i}={p:.2f}' for i, p in enumerate(probs)])
    axes[grade][2].set_xlabel(conf_str, fontsize=7.5, family='monospace')

    for ax in axes[grade]:
        ax.axis('off')
    axes[grade][2].axis('on')
    axes[grade][2].set_xticks([]); axes[grade][2].set_yticks([])

plt.colorbar(im, ax=axes[:, 1], shrink=0.4,
             label='Activation intensity', pad=0.02)
plt.suptitle(
    'Grad-CAM — regions the model focuses on for each DR severity grade\n'
    'Red = high activation (model attention), Blue = low',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'gradcam_all_grades.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gradcam_all_grades.png')

---
## Cell 20 — Grad-CAM vs Grad-CAM++ comparison
Side-by-side comparison on a Moderate DR image. Grad-CAM++ typically produces
a more focused activation around the optic disc and microaneurysm clusters
compared to the broader heatmap of standard Grad-CAM.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 20 — Grad-CAM vs Grad-CAM++ comparison                               ║
# ║  Output: gradcam_vs_gradcampp.png                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Use a Moderate DR example (grade 2) — most interesting clinically
sample = train_df[
    (train_df['diagnosis'] == 2) & (train_df['fold'] == 0)
].iloc[1]    # take the second sample for variety

img_np = load_and_preprocess(sample['path'], CFG['img_size'])

# Run both methods — need separate backward calls so use cloned tensors
cam_v1, pred1, probs1 = gradcam.generate(img_to_tensor(img_np).clone())
cam_v2, pred2, probs2 = gradcampp.generate(img_to_tensor(img_np).clone())

overlay1, _ = overlay_heatmap(img_np, cam_v1)
overlay2, _ = overlay_heatmap(img_np, cam_v2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_np)
axes[0].set_title(f'Original — Grade 2 (Moderate DR)\nTrue grade: {LABEL_MAP[2]}')

axes[1].imshow(overlay1)
axes[1].set_title(f'Grad-CAM overlay\nPred: {LABEL_MAP[pred1]} '
                  f'(conf {probs1[pred1]:.2f})')

axes[2].imshow(overlay2)
axes[2].set_title(f'Grad-CAM++ overlay\nPred: {LABEL_MAP[pred2]} '
                  f'(conf {probs2[pred2]:.2f})\n'
                  '← note tighter focus on lesion areas')

for ax in axes: ax.axis('off')
plt.suptitle('Grad-CAM vs Grad-CAM++ — same model, different weighting strategy',
             fontsize=12)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'gradcam_vs_gradcampp.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: gradcam_vs_gradcampp.png')

---
## Cell 21 — LIME explanation
**LIME** (Local Interpretable Model-agnostic Explanations) works differently
from Grad-CAM — it doesn't require gradient access to the model.
Instead it:
1. Segments the image into superpixels (irregular regions of similar colour)
2. Randomly masks subsets of superpixels
3. Runs each masked image through the model
4. Fits a simple linear model: which superpixels had the most impact?

LIME is **model-agnostic** and can explain any black-box predictor,
making it a useful cross-check against Grad-CAM.

⏱ *This cell takes ~60 seconds with `num_samples=300`.*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 21 — LIME explanation                                                ║
# ║  Ribeiro et al. 2016 https://arxiv.org/abs/1602.04938                     ║
# ║  Model-agnostic: works by perturbing superpixels, not gradients.          ║
# ║  Increase num_samples to 1000 for publication-quality figures (~5 min).   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Prediction function that LIME calls ───────────────────────────────────────
def lime_predict(imgs_np_batch):
    """
    LIME calls this with a numpy array of shape (N, H, W, 3) uint8.
    We normalise, batch them, run through the model, and return
    softmax probabilities of shape (N, num_classes).
    """
    batch = []
    for img in imgs_np_batch:
        x = img.astype(np.float32) / 255.0
        x = (x - MEAN) / STD
        batch.append(torch.from_numpy(x).permute(2, 0, 1).float())

    tensor = torch.stack(batch).to(DEVICE)
    with torch.no_grad():
        logits = xai_model(tensor)
    return F.softmax(logits, dim=1).cpu().numpy()


# ── Run LIME on a Severe DR image (grade 3) ───────────────────────────────────
lime_sample = train_df[
    (train_df['diagnosis'] == 3) & (train_df['fold'] == 0)
].iloc[0]
lime_img = load_and_preprocess(lime_sample['path'], CFG['img_size'])
pred_label = lime_predict(lime_img[np.newaxis]).argmax()

explainer = lime_image.LimeImageExplainer(random_state=CFG['seed'])
print('Running LIME... (~60 seconds with 300 samples)')
explanation = explainer.explain_instance(
    lime_img,
    lime_predict,
    top_labels   = 5,
    hide_color   = 0,
    num_samples  = 300,    # increase to 1000 for publication quality
    random_seed  = CFG['seed']
)
print(f'LIME done. Predicted class: {LABEL_MAP[pred_label]}')

# ── Visualise 4 LIME configurations ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(19, 5))
axes[0].imshow(lime_img)
axes[0].set_title(f'Input (Grade 3 — Severe)\nPred: {LABEL_MAP[pred_label]}')

configs = [
    (True,  False,  5, 'Top-5 positive\n(support prediction)'),
    (True,  False, 10, 'Top-10 positive\n(wider context)'),
    (False, False,  5, 'Positive (green) &\nNegative (red) regions'),
]
for ax, (pos_only, hide, n_feat, title) in zip(axes[1:], configs):
    img_exp, mask = explanation.get_image_and_mask(
        pred_label, positive_only=pos_only,
        num_features=n_feat, hide_rest=hide
    )
    ax.imshow(mark_boundaries(img_exp, mask, color=(0, 1, 0)))
    ax.set_title(title)

for ax in axes: ax.axis('off')
plt.suptitle('LIME — superpixels that drive the Severe DR prediction\n'
             'Green borders = positive contribution, Red = negative',
             fontsize=12)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'lime_explanation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: lime_explanation.png')

---
## Cell 22 — Publication figure generator
Generates a 4-panel figure per image:
`Original → Grad-CAM → Grad-CAM++ → LIME`

This is the **Figure 3** equivalent in your paper. Run it for Grades 2, 3, 4
(the clinically important cases) and pick the most visually interpretable ones.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 22 — Publication-quality 4-panel XAI figures                        ║
# ║  Original | Grad-CAM | Grad-CAM++ | LIME — one figure per DR grade.       ║
# ║  These are the figures that go into your paper / thesis.                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def publication_figure(img_path, true_label, save_path, num_lime_samples=300):
    """
    Generate and save a 4-panel XAI figure for one image.

    Parameters
    ----------
    img_path         : path to the retinal fundus image
    true_label       : int, ground truth grade
    save_path        : where to save the PNG
    num_lime_samples : LIME perturbation samples (300=fast, 1000=publication)
    """
    img_np = load_and_preprocess(img_path, CFG['img_size'])
    tensor = img_to_tensor(img_np)

    # ── Grad-CAM ─────────────────────────────────────────────────────────────
    cam1, pred1, probs = gradcam.generate(tensor.clone())
    overlay1, _        = overlay_heatmap(img_np, cam1)

    # ── Grad-CAM++ ───────────────────────────────────────────────────────────
    cam2, _, _  = gradcampp.generate(tensor.clone())
    overlay2, _ = overlay_heatmap(img_np, cam2)

    # ── LIME ─────────────────────────────────────────────────────────────────
    exp = lime_image.LimeImageExplainer(random_state=CFG['seed']).explain_instance(
        img_np, lime_predict,
        top_labels=5, hide_color=0,
        num_samples=num_lime_samples, random_seed=CFG['seed']
    )
    img_lime, mask = exp.get_image_and_mask(
        pred1, positive_only=True,
        num_features=5, hide_rest=False
    )

    # ── Build figure ─────────────────────────────────────────────────────────
    conf = '  '.join([f'G{i}:{p:.2f}' for i, p in enumerate(probs)])

    fig, axes = plt.subplots(1, 4, figsize=(19, 5))

    axes[0].imshow(img_np)
    axes[0].set_title(
        f'Fundus image\nTrue grade: {true_label} — {LABEL_MAP[true_label]}',
        fontsize=11
    )
    axes[1].imshow(overlay1)
    axes[1].set_title(
        f'Grad-CAM\nPred: {LABEL_MAP[pred1]} (G{pred1})',
        fontsize=11
    )
    axes[2].imshow(overlay2)
    axes[2].set_title('Grad-CAM++\n(finer lesion localisation)', fontsize=11)

    axes[3].imshow(mark_boundaries(img_lime, mask, color=(0.1, 0.9, 0.1)))
    axes[3].set_title('LIME\n(model-agnostic superpixel explanation)', fontsize=11)
    axes[3].set_xlabel(conf, fontsize=7.5, family='monospace')
    axes[3].xaxis.set_tick_params(labelbottom=True)

    for ax in axes:
        ax.axis('off')
    axes[3].axis('on')
    axes[3].set_xticks([]); axes[3].set_yticks([])

    correct = '✓ Correct' if pred1 == true_label else f'✗ Misclassified as {LABEL_MAP[pred1]}'
    plt.suptitle(
        f'XAI output — Grade {true_label} ({LABEL_MAP[true_label]})  |  {correct}',
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


# ── Generate one figure per clinically important grade ────────────────────────
for grade in [0, 2, 3, 4]:   # No DR, Moderate, Severe, Proliferative
    row = train_df[
        (train_df['diagnosis'] == grade) & (train_df['fold'] == 0)
    ].iloc[0]
    publication_figure(
        row['path'],
        grade,
        save_path=CFG['output_dir'] / f'pub_fig_grade{grade}.png'
    )

---
## Cell 23 — Summary of all outputs
Lists every file this notebook saved. These are your deliverables:
- **Figures** → go into your paper / poster / thesis
- **Checkpoints** → saved model weights for reproducibility / deployment

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 23 — Summary of all outputs                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

png_files  = sorted(CFG['output_dir'].glob('**/*.png'))
pth_files  = sorted(CFG['output_dir'].glob('**/*.pth'))

print('═' * 60)
print('  FIGURES (copy to your report / paper)')
print('═' * 60)
for f in png_files:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40s}  {size_kb:>7.1f} KB')

print()
print('═' * 60)
print('  CHECKPOINTS (best model weights per fold)')
print('═' * 60)
for f in pth_files:
    size_mb = f.stat().st_size / (1024**2)
    ckpt    = torch.load(f, map_location='cpu', weights_only=False)
    print(f'  {f.name:<35s}  QWK={ckpt["best_qwk"]:.4f}  '
          f'epoch={ckpt["epoch"]:>2d}  {size_mb:.1f} MB')

if all_qwks:
    print()
    print('═' * 60)
    print('  CROSS-VALIDATION RESULTS')
    print('═' * 60)
    for i, q in enumerate(all_qwks):
        print(f'  Fold {i}: QWK = {q:.4f}')
    print(f'  Mean  : QWK = {np.mean(all_qwks):.4f} ± {np.std(all_qwks):.4f}')

print()
print('✓ Notebook complete!')
print()
print('Next steps:')
print('  1. Remove the `break` in Cell 14 and run all 5 folds')
print('  2. Increase img_size to 380 in CFG for better accuracy')
print('  3. Increase LIME num_samples to 1000 for publication figures')
print('  4. Try backbone="densenet121" or "resnet50" for ablation table')
print('  5. Use the pub_fig_grade*.png files as Figure 3 in your paper')